In [1]:
from __future__ import annotations

import os
import json
from pathlib import Path
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
PROJECT_ROOT = Path("..").resolve()

print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())

/home/lipov/projects/Python_practice_sessions_05
True


In [5]:
ALLOWED_EXTENSIONS = {".md", ".py", ".txt", ".toml"}

EXCLUDED_DIRS = {
    ".git",
    ".venv",
    "__pycache__",
    ".mypy_cache",
    ".pytest_cache",
    ".ruff_cache",
    ".ipynb_checkpoints",
    "images",
    "test_images",
    "htmlcov",
    "dist",
    "build",
}


def should_skip_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIRS for part in path.parts)


def collect_text_files(root: Path) -> list[Path]:
    files = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if should_skip_path(path):
            continue

        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        files.append(path)

    return sorted(files)

In [6]:
files = collect_text_files(PROJECT_ROOT)

print("Files found:", len(files))

for path in files[:30]:
    print(path.relative_to(PROJECT_ROOT))

Files found: 536
Contex_manager_samples/__init__.py
Contex_manager_samples/sample_1_.py
Contex_manager_samples/sample_2_.py
Contex_manager_samples/sample_3_.py
Contex_manager_samples/sample_4_.py
Functional_Programming/__init__.py
Functional_Programming/closure_.py
Functional_Programming/closure_exercises.py
Functional_Programming/closure_practice_.py
Functional_Programming/closure_practice_0.py
Functional_Programming/composition_.py
Functional_Programming/decorators_.py
Functional_Programming/decorators_2.py
Functional_Programming/decorators_practice.py
Functional_Programming/example_3.py
Functional_Programming/monada.py
Functional_Programming/setter_getter_ex_1.py
Functional_Programming/setter_getter_ex_2.py
Neural_Networks/ex_prototype.py
Neural_Networks/ex_prototype_description.py
Neural_Networks/impl_perc.py
Neural_Networks/perceptron.py
OOP_Programming/__init__.py
OOP_Programming/attributes_/__init__.py
OOP_Programming/attributes_/attrs_.py
OOP_Programming/attributes_/attrs_looku

In [7]:
def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [8]:
def chunk_text(text: str, *, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    text = text.strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [9]:
def build_chunks(files: list[Path], root: Path) -> list[dict]:
    records = []

    for path in files:
        text = read_text_file(path)
        chunks = chunk_text(text)

        relative_path = str(path.relative_to(root))

        for index, chunk in enumerate(chunks):
            records.append(
                {
                    "content": chunk,
                    "source": relative_path,
                    "file_type": path.suffix.lower().lstrip("."),
                    "chunk_index": index,
                }
            )

    return records

In [10]:
records = build_chunks(files, PROJECT_ROOT)

print("Chunks:", len(records))
print(records[0])

Chunks: 2041
{'content': "class Yolo:\n    def __init__(self):\n        print('init')\n\n    def __enter__(self):\n        print('entering')\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb):\n        if exc_type:\n            print('exception')\n        else:\n            print('exiting')\n        return True\n\n\nwith Yolo() as y:\n    print('normal code')\n    raise Exception('oops')\n\nprint('Next code')", 'source': 'Contex_manager_samples/sample_1_.py', 'file_type': 'py', 'chunk_index': 0}


In [11]:
COLLECTION_NAME = "ProjectChunk"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

project_chunks = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="file_type",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="chunk_index",
            data_type=wvc.config.DataType.INT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: ProjectChunk


In [12]:
result = project_chunks.data.insert_many(records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [13]:
count = project_chunks.aggregate.over_all(total_count=True)

print("Objects in collection:", count.total_count)

Objects in collection: 2041


In [14]:
response = project_chunks.query.fetch_objects(
    limit=5,
    return_properties=["source", "file_type", "chunk_index", "content"],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Source:", obj.properties["source"])
    print("Type:", obj.properties["file_type"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:300])
    print("-" * 80)

UUID: 0001d410-e0bd-4a7b-9a92-0375b9842161
Source: demos/random-stuff/teaser_dragon_ctf_2019/playcap/test/gamepad.txt
Type: txt
Chunk: 104
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
1568814380.349384000	30c79100800178e87b28a87a0a000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
1568814380.365248000	30c99100800176f87b27a87a0a000
--------------------------------------------------------------------------------
UUID: 005906ca-da58-4285-9cf8-79b7013d4a52
Source: demos/random-stuff/teaser_dragon_ctf_2019/playcap/test/gamepad.txt
Type: txt
Chunk: 570
1568814421.837260000	304c9100800078c87b27987a0a000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
1568814421.853355000	30509100800077d87b27a87a0b000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

---------------------------------------

In [15]:
query = "How does the project connect to Weaviate Cloud?"

response = project_chunks.query.near_text(
    query=query,
    limit=5,
    return_properties=["source", "file_type", "chunk_index", "content"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Source:", obj.properties["source"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:600])
    print("-" * 80)

Distance: 0.7780277132987976
Source: README.md
Chunk: 1
notebooks (`Jupyter Notebook`)
- Managing dependencies using `uv`

# uv

An extremely fast Python package and project manager, written in Rust.

![uv Benchmark](https://github.com/astral-sh/uv/assets/1309177/03aa9163-1c79-4a87-a31d-7a9311ed9310#only-dark)

## Highlights

- 🚀 A single tool to replace `pip`, `pip-tools`, `pipx`, `poetry`, `pyenv`, `twine`, `virtualenv`, and more.
- ⚡️ 10-100x faster than `pip`.
- 🐍 Installs and manages Python versions.
- 🛠️ Runs and installs Python applications.
- ❇️ Runs scripts, with support for inline dependency metadata.
- 🗂️ Provides comprehensive project m
--------------------------------------------------------------------------------
Distance: 0.7794193625450134
Source: README.md
Chunk: 3
```bash
   pipx install uv
   ```

For Linux and macOS, determine your shell (e.g., with `echo $SHELL`), then run one of:

```bash
# For Bash
echo 'eval "$(uv generate-shell-completion bash)"' >> ~/.bashrc


In [16]:
query = "text to image search CLIP embeddings"

response = project_chunks.query.hybrid(
    query=query,
    alpha=0.5,
    limit=5,
    return_properties=["source", "file_type", "chunk_index", "content"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Source:", obj.properties["source"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:600])
    print("-" * 80)

Score: 0.7390622496604919
Source: demos/random-stuff/text-to-speech/tts_worker.py
Chunk: 1
ynthesisInput(
      text=text
  )

  voice_params = texttospeech.VoiceSelectionParams(
      language_code=language_code, name=VOICE_NAME
  )

  audio_config = texttospeech.AudioConfig(
      audio_encoding=texttospeech.AudioEncoding.MP3,
      speaking_rate=SPEAKING_RATE,
      pitch=VOICE_PITCH
  )

  # TODO: Perhaps it's enough to keep a reference to the client?
  client = texttospeech.TextToSpeechClient()

  response = client.synthesize_speech(
      input=text_input,
      voice=voice_params,
      audio_config=audio_config
  )

  with open(fname, "wb") as out:
    out.write(response.aud
--------------------------------------------------------------------------------
Score: 0.550987184047699
Source: demos/random-stuff/text-to-speech/README.md
Chunk: 0
# Text file to MP3 converter (using Google Cloud API)

**Fair warning - this is a very simple hacked up script.**

**Use at your own risk.**


In [17]:
query = "function that connects to Weaviate"

response = project_chunks.query.near_text(
    query=query,
    filters=Filter.by_property("file_type").equal("py"),
    limit=5,
    return_properties=["source", "file_type", "chunk_index", "content"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Source:", obj.properties["source"])
    print(obj.properties["content"][:600])
    print("-" * 80)

Distance: 0.7179299592971802
Source: demos/exercises_2/client_api_oop.py
rn f"<JourneyLeg {self.mode}: {self.origin} → {self.destination} ({self.departure} → {self.arrival})>"


class VBBTransportAPI:
    BASE_URL = "https://v6.vbb.transport.rest"

    def __init__(self):
        self.session = requests.Session()

    def _get(self, endpoint: str, params: dict = None) -> dict | list:
        url = f"{self.BASE_URL}{endpoint}"
        response = self.session.get(url, params=params)
        response.raise_for_status()
        return response.json()

    def search_locations(self, query: str, limit: int = 5) -> list[Location]:
        data = self._get("/locations", {"
--------------------------------------------------------------------------------
Distance: 0.7194448113441467
Source: demos/exercises_2/busstops_finder_gui.py
import requests
import streamlit as st
from streamlit_folium import st_folium
import folium


class VBBTransportAPI:
    BASE_URL = "https://v6.vbb.transport.rest"

  

In [18]:
query = "Explain how Weaviate Cloud is used in this project."

response = project_chunks.generate.near_text(
    query=query,
    grouped_task=(
        "Answer the user's question using only the retrieved project context. "
        "Keep the answer concise and practical. "
        "Mention the relevant source files if possible."
    ),
    limit=6,
    return_properties=["source", "chunk_index", "content"],
)

print(response.generated)

To manage dependencies in your Python projects, you can use the `uv` package. Here’s how to get started:

1. **Install `uv`**: Use `pipx install uv` to install it.
2. **Initialize `uv` for your project**: Run `uv init` to set it up.
3. **Add dependencies**: Use `uv add mypy ruff pytest --dev` to include development dependencies.
4. **Activate the virtual environment**: Run `source .venv/bin/activate`.

For more details, check the `README.md` in your project directory.


In [19]:
print("Sources used:")

for obj in response.objects:
    print(
        f"- {obj.properties['source']} "
        f"(chunk {obj.properties['chunk_index']})"
    )

Sources used:
- README.md (chunk 1)
- README.md (chunk 3)
- demos/random-stuff/text-to-speech/README.md (chunk 0)
- OOP_Programming/employees_it.md (chunk 1)
- README.md (chunk 0)
- demos/random-stuff/teaser_dragon_ctf_2019/playcap/README.md (chunk 0)


In [20]:
def ask_project(question: str, *, limit: int = 6) -> None:
    response = project_chunks.generate.near_text(
        query=question,
        grouped_task=(
            "Answer the user's question using only the retrieved project context. "
            "If the context is not enough, say that the project context is insufficient. "
            "Keep the answer concise."
        ),
        limit=limit,
        return_properties=["source", "chunk_index", "content"],
    )

    print("ANSWER:")
    print(response.generated)

    print("\nSOURCES:")
    for obj in response.objects:
        print(
            f"- {obj.properties['source']} "
            f"(chunk {obj.properties['chunk_index']})"
        )

In [21]:
ask_project("What is this repository about?")

ANSWER:
The project context is insufficient.

SOURCES:
- demos/random-stuff/README.md (chunk 0)
- README.md (chunk 2)
- md/git_commands.md (chunk 4)
- demos/exercises_6/dir_walk/weird-dir-name-list.txt (chunk 2)
- demos/exercises_6/dir_walk/weird-dir-name-list.txt (chunk 3)
- demos/random-stuff/gosu-src/README.md (chunk 0)


In [22]:
ask_project("Which files or notebooks are related to Weaviate Cloud?")

ANSWER:
The project context is insufficient.

SOURCES:
- README.md (chunk 1)
- demos/exercises_6/dir_walk/weird-dir-name-list.txt (chunk 3)
- demos/exercises_6/dir_walk/weird-dir-name-list.txt (chunk 0)
- demos/exercises_6/dir_walk/weird-dir-name-list.txt (chunk 4)
- demos/exercises_1/output_A.txt (chunk 5)
- demos/exercises_1/output_B.txt (chunk 5)


In [23]:
ask_project("Explain the difference between vector search, hybrid search and generative search.")

ANSWER:
The project context is insufficient.

SOURCES:
- demos/random-stuff/pypypypy-genetic/README.md (chunk 0)
- demos/random-stuff/pypypypy-genetic/opcodes.py (chunk 2)
- demos/random-stuff/pypypypy-genetic/opcodes.py (chunk 3)
- demos/random-stuff/pypypypy-genetic/vis.py (chunk 3)
- demos/random-stuff/pypypypy-genetic/vis.py (chunk 5)
- demos/exercises_1/output_B.txt (chunk 40)


In [24]:
ask_project("Where is text-to-image search implemented?")

ANSWER:
The project context is insufficient.

SOURCES:
- demos/random-stuff/bookmarklets/README.md (chunk 0)
- demos/random-stuff/text-to-speech/tts_worker.py (chunk 1)
- demos/random-stuff/text-to-speech/README.md (chunk 0)
- demos/exercises_1/output_B.txt (chunk 29)
- demos/exercises_1/output_C.txt (chunk 48)
- demos/exercises_1/output_A.txt (chunk 48)


In [25]:
NOTEBOOK_ROOT = Path("../vector_db").resolve()

print(NOTEBOOK_ROOT)
print(NOTEBOOK_ROOT.exists())

/home/lipov/projects/Python_practice_sessions_05/vector_db
True


In [26]:
ALLOWED_EXTENSIONS = {".ipynb", ".md", ".py", ".txt"}

EXCLUDED_DIRS = {
    ".ipynb_checkpoints",
    "__pycache__",
}

In [27]:
def read_ipynb_file(path: Path) -> str:
    data = json.loads(path.read_text(encoding="utf-8"))

    parts = []

    for cell in data.get("cells", []):
        cell_type = cell.get("cell_type", "")
        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        parts.append(f"[{cell_type.upper()} CELL]\n{source}")

    return "\n\n".join(parts)

In [28]:
def read_file(path: Path) -> str:
    if path.suffix.lower() == ".ipynb":
        return read_ipynb_file(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [29]:
def should_skip_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIRS for part in path.parts)


def collect_files(root: Path) -> list[Path]:
    files = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if should_skip_path(path):
            continue

        if path.suffix.lower() not in ALLOWED_EXTENSIONS:
            continue

        files.append(path)

    return sorted(files)

In [30]:
files = collect_files(NOTEBOOK_ROOT)

print("Files found:", len(files))

for path in files:
    print(path.relative_to(NOTEBOOK_ROOT))

Files found: 12
01_vectors.ipynb
02_txt_to_image.ipynb
03_crud.ipynb
04_hybrid_search.ipynb
05_generative_search.ipynb
06_veawiate_cloud.ipynb
07_w_cluster.ipynb
08_w_cluster_hybrid_search.ipynb
09_w_cluster_vectors.ipynb
10_w_cluster_txt_to_image.ipynb
11_w_cluster_generative_search.ipynb
12_11_w_cluster_rag_project_docs.ipynb


In [31]:
def chunk_text(text: str, *, chunk_size: int = 1500, overlap: int = 250) -> list[str]:
    text = text.strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [32]:
def build_chunks(files: list[Path], root: Path) -> list[dict]:
    records = []

    for path in files:
        text = read_file(path)
        chunks = chunk_text(text)

        relative_path = str(path.relative_to(root))

        for index, chunk in enumerate(chunks):
            records.append(
                {
                    "content": chunk,
                    "source": relative_path,
                    "file_type": path.suffix.lower().lstrip("."),
                    "chunk_index": index,
                }
            )

    return records

In [33]:
records = build_chunks(files, NOTEBOOK_ROOT)

print("Chunks:", len(records))
print(records[0]["source"])
print(records[0]["content"][:500])

Chunks: 61
01_vectors.ipynb
[CODE CELL]
from __future__ import annotations
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

[CODE CELL]
sns.set_theme(style='whitegrid', context='notebook')

[CODE CELL]
color_1 = [40, 120, 60]
color_2 = [60, 50, 90]

[CODE CELL]
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")

ax.scatter(
    xs=color_1[0],
    ys=color_1[1],
    zs=color_1[2],
    label="Color 1",
    color="royalblue",
    s=50,
)

ax.scatter(
    xs=color_2[0],
    ys=color


In [34]:
COLLECTION_NAME = "WeaviateNotebookChunk"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

notebook_chunks = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="file_type",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="chunk_index",
            data_type=wvc.config.DataType.INT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: WeaviateNotebookChunk


In [35]:
result = notebook_chunks.data.insert_many(records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [36]:
count = notebook_chunks.aggregate.over_all(total_count=True)

print("Objects:", count.total_count)

Objects: 61


In [37]:
query = "text-to-image search CLIP embeddings images test_images"

response = notebook_chunks.query.hybrid(
    query=query,
    alpha=0.3,
    limit=8,
    return_properties=["source", "chunk_index", "content"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Source:", obj.properties["source"])
    print("Chunk:", obj.properties["chunk_index"])
    print(obj.properties["content"][:700])
    print("-" * 80)

Score: 0.9201616644859314
Source: 10_w_cluster_txt_to_image.ipynb
Chunk: 0
[CODE CELL]
import os
from pathlib import Path
from getpass import getpass

from PIL import Image
from IPython.display import Image as IPythonImage, display
from sentence_transformers import SentenceTransformer

import weaviate
import weaviate.classes as wvc
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

[CODE CELL]
IMAGE_DIR = Path("./images")
TEST_IMAGE_DIR = Path("./test_images")

print("Images:", IMAGE_DIR.exists())
print("Test images:", TEST_IMAGE_DIR.exists())

[CODE CELL]
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] =
--------------------------------------------------------------------------------
Score: 0.9047070145606995
Source: 10_w_cluster_txt_to_image.ipynb
Chunk: 1
[float]:
    image = Image.open(path).convert("RGB")
    vector = cl

In [38]:
def ask_notebooks(question: str, *, limit: int = 8) -> None:
    response = notebook_chunks.generate.hybrid(
        query=question,
        alpha=0.3,
        grouped_task=(
            "Answer the user's question using only the retrieved notebook context. "
            "If the context is not enough, say that the notebook context is insufficient. "
            "Mention the most relevant notebook file names."
        ),
        limit=limit,
        return_properties=["source", "chunk_index", "content"],
    )

    print("ANSWER:")
    print(response.generated)

    print("\nSOURCES:")
    for obj in response.objects:
        print(
            f"- {obj.properties['source']} "
            f"(chunk {obj.properties['chunk_index']})"
        )

In [39]:
ask_notebooks("Where is text-to-image search implemented?")

ANSWER:
The notebook context is insufficient to answer your question. The most relevant notebook files are: 

1. `10_w_cluster_txt_to_image.ipynb`
2. `02_txt_to_image.ipynb`
3. `12_11_w_cluster_rag_project_docs.ipynb`

SOURCES:
- 12_11_w_cluster_rag_project_docs.ipynb (chunk 6)
- 10_w_cluster_txt_to_image.ipynb (chunk 1)
- 10_w_cluster_txt_to_image.ipynb (chunk 3)
- 10_w_cluster_txt_to_image.ipynb (chunk 2)
- 10_w_cluster_txt_to_image.ipynb (chunk 0)
- 02_txt_to_image.ipynb (chunk 2)
- 02_txt_to_image.ipynb (chunk 1)
- 02_txt_to_image.ipynb (chunk 0)


In [40]:
ask_notebooks("Which notebook implements text-to-image search with CLIP embeddings and Weaviate Cloud?")

ANSWER:
The notebook context is insufficient to answer your question. The most relevant notebook files are:

1. **10_w_cluster_txt_to_image.ipynb**
2. **02_txt_to_image.ipynb**
3. **12_11_w_cluster_rag_project_docs.ipynb**
4. **07_w_cluster.ipynb**
5. **06_veawiate_cloud.ipynb**

SOURCES:
- 10_w_cluster_txt_to_image.ipynb (chunk 0)
- 10_w_cluster_txt_to_image.ipynb (chunk 1)
- 12_11_w_cluster_rag_project_docs.ipynb (chunk 6)
- 02_txt_to_image.ipynb (chunk 0)
- 12_11_w_cluster_rag_project_docs.ipynb (chunk 4)
- 02_txt_to_image.ipynb (chunk 1)
- 07_w_cluster.ipynb (chunk 5)
- 06_veawiate_cloud.ipynb (chunk 0)


In [41]:
ask_notebooks("Explain what is the differnece between bm25 and vector search?")

ANSWER:
The notebook context is insufficient to answer your question. However, the most relevant notebook files are:

1. **08_w_cluster_hybrid_search.ipynb**
2. **04_hybrid_search.ipynb**
3. **12_11_w_cluster_rag_project_docs.ipynb**
4. **07_w_cluster.ipynb**
5. **09_w_cluster_vectors.ipynb**

SOURCES:
- 08_w_cluster_hybrid_search.ipynb (chunk 2)
- 04_hybrid_search.ipynb (chunk 2)
- 12_11_w_cluster_rag_project_docs.ipynb (chunk 6)
- 08_w_cluster_hybrid_search.ipynb (chunk 6)
- 07_w_cluster.ipynb (chunk 5)
- 08_w_cluster_hybrid_search.ipynb (chunk 7)
- 08_w_cluster_hybrid_search.ipynb (chunk 3)
- 09_w_cluster_vectors.ipynb (chunk 3)


In [42]:
client.close()